# Ozon E-CUP 2026 — E5 Macro V2

Цель этого эксперимента — сделать **сильный независимый cross-encoder**, который имеет шанс заметно превзойти `rubert-tiny2 stage-A`.

Что меняем относительно текущего CE:

1. **Более сильный backbone:** `intfloat/multilingual-e5-small`.
2. **Только LLM stage-A:** ручной fine-tune не используется, потому что по командным A/B-тестам он ухудшал leaderboard.
3. **Macro-aware обучение:** категории получают веса, приближающие objective к macro PR-AUC.
4. **Небольшой дополнительный boost слабых fashion-категорий:** Обувь / Одежда / Галантерея / Ювелирные украшения.
5. **Confidence-aware soft BCE:** неоднозначные LLM-метки около 0.5 слегка downweight-ятся, но не выбрасываются.
6. **Random swap:** во время обучения товары случайно меняются местами, чтобы cross-encoder меньше зависел от порядка пары.
7. **Улучшенный текст товара:** категория явно добавляется в текст; важные атрибуты получают приоритет; лимит атрибутов увеличен.
8. **Best checkpoint:** сохраняется именно лучший checkpoint по групповому LLM holdout, а не просто последняя модель.
9. **Тот же group split**, чтобы метрики были сопоставимы с предыдущими экспериментами команды.

Главная метрика для принятия решения — **LLM group holdout Macro PR-AUC на уверенных LLM-метках**.  
Manual holdout считаем только как диагностику и **не используем для выбора модели**.

> Практический ориентир команды: предыдущий `rubert-tiny2 stage-A` дал LB ≈ 0.450.  
> Этот ноутбук задуман как один сильный эксперимент на сегодня, а не как гарантированный 0.50.

**Перед запуском:** вручную выбери нужный GPU и Internet в Settings Kaggle. Сам ноутбук больше не содержит Kaggle-specific metadata и не меняет эти настройки. Если выбранный GPU даст CUDA OOM, уменьши `MICRO_BATCH` и увеличь `GRAD_ACCUM`, сохраняя effective batch около 256.


In [1]:
import os
import gc
import json
import math
import time
import random
import shutil
import warnings
import re
from collections import Counter

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")

# -----------------------------
# Paths
# -----------------------------
BASE = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"

ITEMS_PATH = f"{BASE}/items.parquet"
ITEMS_HUMAN_PATH = f"{BASE}/items_human.parquet"
MATCHES_PATH = f"{BASE}/matches.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"

# -----------------------------
# Experiment
# -----------------------------
MODEL_NAME = "intfloat/multilingual-e5-small"
OUT_DIR = "/kaggle/working/e5_macro_v2_final"
BEST_CKPT = "/kaggle/working/e5_macro_v2_best.pt"
METRICS_PATH = "/kaggle/working/e5_macro_v2_metrics.json"

SEED = 42

# Current CE used 160 tokens + only 260 attr chars.
# Here we keep inference reasonably short, but feed more useful structured text.
MAX_LEN = 192
MAX_ATTR_CHARS = 460

# One full pass over LLM pairs.
EPOCHS = 1

# LR is intentionally much lower than 2e-4 used for rubert-tiny2.
BACKBONE_LR = 3e-5
HEAD_LR = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

# Macro-aligned weighting
USE_CATEGORY_BALANCE = True
CATEGORY_WEIGHT_POWER = 0.50
CATEGORY_WEIGHT_MIN = 0.65
CATEGORY_WEIGHT_MAX = 2.00

# Extra small push for the four categories that were consistently weakest.
FASHION_BOOST = 1.15
FASHION_CATEGORIES = {
    "Обувь",
    "Одежда",
    "Галантерея",
    "Ювелирные украшения",
    "Ювелирка",  # harmless fallback if category naming differs
}

# LLM soft-label confidence weighting.
# 0.5-label still receives 75% of normal weight; 0/1 receives 100%.
USE_CONFIDENCE_WEIGHT = True
CONFIDENCE_FLOOR = 0.75
CONFIDENCE_POWER = 1.0

RANDOM_SWAP_PROB = 0.50

# Validation
LLM_VAL_FRAC = 0.03
LLM_VAL_SEED = 13
MANUAL_VAL_FRAC = 0.20
MANUAL_VAL_SEED = 42

FAST_VAL_PER_CATEGORY = 3000
EVAL_EVERY_OPT_STEPS = 10000

# Set True only if memory is tight.
USE_GRADIENT_CHECKPOINTING = False

# -----------------------------
# Reproducibility / GPU
# -----------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "Для этого эксперимента включи GPU Accelerator в Kaggle."

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Conservative automatic batch profile.
if gpu_mem_gb >= 35:
    MICRO_BATCH = 128
    GRAD_ACCUM = 2
elif gpu_mem_gb >= 15:
    # Safe default for Kaggle T4 16 GB.
    MICRO_BATCH = 64
    GRAD_ACCUM = 4
else:
    MICRO_BATCH = 32
    GRAD_ACCUM = 8

EFFECTIVE_BATCH = MICRO_BATCH * GRAD_ACCUM
PRED_BATCH = MICRO_BATCH * 2

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

print("GPU:", gpu_name)
print(f"GPU memory: {gpu_mem_gb:.1f} GB")
print("micro batch:", MICRO_BATCH)
print("grad accum:", GRAD_ACCUM)
print("effective batch:", EFFECTIVE_BATCH)
print("max_len:", MAX_LEN)

GPU: Tesla T4
GPU memory: 14.6 GB
micro batch: 32
grad accum: 8
effective batch: 256
max_len: 192


## 1. Проверка файлов

Этот cell падает сразу, если какой-то Kaggle dataset path указан неверно.

In [2]:
required_paths = [
    ITEMS_PATH,
    ITEMS_HUMAN_PATH,
    MATCHES_PATH,
    MATCHES_LLM_PATH,
]

for p in required_paths:
    assert os.path.exists(p), f"Не найден файл: {p}"
    print(f"{os.path.basename(p):24s} {os.path.getsize(p) / 1024**3:8.3f} GB")

items.parquet               3.822 GB
items_human.parquet         0.199 GB
matches.parquet             0.004 GB
matches_llm.parquet         0.098 GB


## 2. Текст товара V2

Ключевая правка относительно старого CE: категория теперь явно присутствует в тексте.

Также не полагаемся на случайный порядок JSON-атрибутов: сначала помещаем поля, которые чаще всего решают identity товара — бренд, модель, артикул/OEM, размеры, цвет, материал, объём/вес и т.д.

Важно: раньше `MAX_LEN=160`, но строка атрибутов предварительно обрезалась до ~260 символов. Поэтому простое увеличение `MAX_LEN` само по себе могло почти ничего не дать. Здесь одновременно увеличиваем полезный character budget и слегка увеличиваем token budget.

In [3]:
SPACE_RE = re.compile(r"\s+")
MULTIPLY_RE = re.compile(r"[×хХ]")

KEY_ORDER = [
    # exact identity / codes
    "бренд", "brand",
    "артикул", "партномер", "part number", "partnumber", "oem",
    "код", "sku", "модель", "model",
    # fashion / variant-defining
    "размер", "size", "рост", "обхват", "пол", "gender",
    "цвет", "color", "материал", "material", "сезон",
    # quantities / dimensions
    "объем", "обьем", "volume", "вес", "weight",
    "длина", "ширина", "высота",
    "количество", "комплектация", "упаков",
    # generic type
    "тип", "type",
]

def normalize_piece(x):
    if x is None:
        return ""
    s = str(x).lower().replace("ё", "е")
    s = MULTIPLY_RE.sub("x", s)
    s = s.replace(",", ".")
    s = SPACE_RE.sub(" ", s).strip()
    return s

def safe_attrs(attributes):
    if isinstance(attributes, dict):
        obj = attributes
    elif isinstance(attributes, str):
        try:
            obj = json.loads(attributes)
        except Exception:
            obj = {}
    else:
        obj = {}

    if not isinstance(obj, dict):
        return {}

    out = {}
    for k, v in obj.items():
        kk = normalize_piece(k)
        vv = normalize_piece(v)
        if kk and vv:
            out[kk] = vv
    return out

def build_text_v2(name, attributes, category, max_attr_chars=MAX_ATTR_CHARS):
    cat = normalize_piece(category)
    nm = normalize_piece(name)
    attrs = safe_attrs(attributes)

    picked = []
    used = set()

    for want in KEY_ORDER:
        for k, v in attrs.items():
            if k in used:
                continue
            if want in k:
                picked.append(f"{k}: {v}")
                used.add(k)

    rest = [f"{k}: {v}" for k, v in attrs.items() if k not in used]
    attr_text = " ; ".join(picked + rest)[:max_attr_chars]

    # Category is explicit context for the matching rule.
    return f"категория: {cat} | название: {nm} | атрибуты: {attr_text}"

# Small sanity check
print(build_text_v2(
    "Кроссовки мужские, размер 42",
    json.dumps({"Бренд": "Test", "Цвет": "Черный", "Размер": "42", "Материал": "Кожа"},
               ensure_ascii=False),
    "Обувь",
))

категория: обувь | название: кроссовки мужские. размер 42 | атрибуты: бренд: test ; размер: 42 ; цвет: черный ; материал: кожа


## 3. Загружаем все товары потоково

Используем тот же подход, что уже работал у команды: читаем большой `items.parquet` батчами через PyArrow и строим словари `id → text` и `id → category`.

Это самая RAM-heavy часть ноутбука. Не создавай параллельно несколько копий `items`.

In [4]:
t0 = time.time()

item_text = {}
item_cat = {}

pf = pq.ParquetFile(ITEMS_PATH)
for batch_id, batch in enumerate(
    pf.iter_batches(
        columns=["id", "name", "attributes", "category"],
        batch_size=400_000,
    ),
    start=1,
):
    pdf = batch.to_pandas()
    for i, n, a, c in pdf.itertuples(index=False, name=None):
        item_text[i] = build_text_v2(n, a, c)
        item_cat[i] = c

    if batch_id % 5 == 0:
        print(
            f"batches={batch_id:3d} items={len(item_text):,} "
            f"time={time.time()-t0:.0f}s",
            flush=True,
        )
    del pdf, batch
    gc.collect()

print(f"Loaded items: {len(item_text):,} in {(time.time()-t0)/60:.1f} min")

batches=  5 items=2,000,000 time=249s


batches= 10 items=4,000,000 time=453s


batches= 15 items=6,000,000 time=649s


batches= 20 items=8,000,000 time=907s


batches= 25 items=10,000,000 time=1152s


batches= 30 items=12,000,000 time=1392s


Loaded items: 13,397,761 in 26.7 min


## 4. Фиксированные group splits

Функция оставлена совместимой с текущим командным CE: union-find по товарам.  
Товар не может одновременно попасть в train и validation.

LLM holdout затем фильтруется до уверенных soft labels (`<= 0.2` или `>= 0.8`) и бинаризуется **только для расчёта метрики**. Train сохраняет исходные soft labels.

In [5]:
def group_val_mask(df, val_frac, seed):
    parent = {}

    def find(x):
        p = parent.setdefault(x, x)
        while p != parent[p]:
            parent[p] = parent[parent[p]]
            p = parent[p]
        parent[x] = p
        return p

    for a, b in zip(df.id1.values, df.id2.values):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    comp = np.fromiter(
        (find(i) for i in df.id1.values),
        dtype=np.int64,
        count=len(df),
    )

    rng = np.random.RandomState(seed)
    uniq = np.unique(comp)
    val_set = set(uniq[rng.rand(len(uniq)) < val_frac].tolist())

    return np.fromiter(
        (c in val_set for c in comp),
        dtype=bool,
        count=len(df),
    )

# Manual
m = pd.read_parquet(MATCHES_PATH)
m_val_mask = group_val_mask(m, MANUAL_VAL_FRAC, MANUAL_VAL_SEED)
m_train = m[~m_val_mask].copy()
m_val = m[m_val_mask].copy()

# LLM
ml = pd.read_parquet(MATCHES_LLM_PATH)
l_val_mask = group_val_mask(ml, LLM_VAL_FRAC, LLM_VAL_SEED)
ml_train = ml[~l_val_mask].copy()
ml_val_raw = ml[l_val_mask].copy()

# Metric holdout: only confident LLM labels
ml_val = ml_val_raw[
    (ml_val_raw.target <= 0.2) | (ml_val_raw.target >= 0.8)
].copy()
ml_val["target"] = (ml_val["target"] >= 0.5).astype(np.int8)

# Categories are required only for validation tables.
m_val["category"] = [item_cat[i] for i in m_val.id1.values]
ml_val["category"] = [item_cat[i] for i in ml_val.id1.values]

print(f"manual train/val: {len(m_train):,} / {len(m_val):,}")
print(f"llm train:        {len(ml_train):,}")
print(f"llm val raw:      {len(ml_val_raw):,}")
print(f"llm val confident:{len(ml_val):,}")

del m, ml, m_val_mask, l_val_mask, ml_val_raw
gc.collect()

manual train/val: 292,706 / 72,948
llm train:        10,950,394
llm val raw:      237,386
llm val confident:191,555


0

## 5. Macro-aware category weights

Соревнование усредняет PR-AUC по 20 категориям, а обычный BCE усредняет loss по парам. Если категории представлены в train неравномерно, эти objectives расходятся.

Используем мягкое inverse-frequency weighting (`power=0.5`) с clipping, а не агрессивный `1 / count`.  
Дополнительно слабые fashion-категории получают небольшой множитель `1.15`.

Это **не oversampling**: длина эпохи не растёт, меняется только вклад примеров в loss.

In [6]:
cat_counts = Counter(item_cat[i] for i in ml_train.id1.values)

counts_arr = np.asarray(list(cat_counts.values()), dtype=np.float64)
reference_count = np.median(counts_arr)

category_weight = {}
for cat, cnt in cat_counts.items():
    w = (reference_count / max(cnt, 1)) ** CATEGORY_WEIGHT_POWER

    if cat in FASHION_CATEGORIES:
        w *= FASHION_BOOST

    w = float(np.clip(w, CATEGORY_WEIGHT_MIN, CATEGORY_WEIGHT_MAX))
    category_weight[cat] = w

# Normalize so average train weight is ~1.
weighted_sum = sum(cat_counts[c] * category_weight[c] for c in cat_counts)
normalizer = weighted_sum / sum(cat_counts.values())
category_weight = {c: w / normalizer for c, w in category_weight.items()}

ml_train["_cat_weight"] = np.fromiter(
    (category_weight[item_cat[i]] for i in ml_train.id1.values),
    dtype=np.float32,
    count=len(ml_train),
)

weight_table = pd.DataFrame({
    "category": list(cat_counts.keys()),
    "pairs": [cat_counts[c] for c in cat_counts],
    "weight": [category_weight[c] for c in cat_counts],
    "fashion": [c in FASHION_CATEGORIES for c in cat_counts],
}).sort_values("weight", ascending=False).reset_index(drop=True)

display(weight_table)
print("mean pair weight:", np.average(
    weight_table["weight"],
    weights=weight_table["pairs"],
))

,category,pairs,weight,fashion
0,Обувь,582863,1.097755,True
1,Одежда,584258,1.096443,True
2,Музыкальные инструменты,513437,1.017061,False
3,Бытовая химия,514820,1.015694,False
4,Ювелирные изделия,522011,1.008674,False
5,Спорт и отдых,534611,0.996717,False
6,Галантерея и аксессуары,538595,0.993023,False
7,Хобби и творчество,538604,0.993015,False
8,Бытовая техника,540156,0.991588,False
9,Товары для животных,540511,0.991262,False


mean pair weight: 1.0000000000000002


## 6. Dataset, random swap и weighted soft BCE

`RANDOM_SWAP_PROB=0.5` делает модель ближе к симметричной функции пары без удвоения датасета.

Confidence weight очень мягкий:
- LLM target = 0.0 / 1.0 → вес 1.00
- LLM target = 0.5 → вес 0.75

То есть неопределённые LLM пары не выбрасываются.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDataset(Dataset):
    def __init__(self, pairs_df, training=False):
        self.id1 = pairs_df.id1.values
        self.id2 = pairs_df.id2.values
        self.y = pairs_df.target.values.astype(np.float32)
        self.training = training

        if "_cat_weight" in pairs_df.columns:
            self.cat_w = pairs_df["_cat_weight"].values.astype(np.float32)
        else:
            self.cat_w = np.ones(len(pairs_df), dtype=np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        a = self.id1[idx]
        b = self.id2[idx]

        if self.training and random.random() < RANDOM_SWAP_PROB:
            a, b = b, a

        y = self.y[idx]
        w = self.cat_w[idx]

        return item_text[a], item_text[b], y, w

def collate(batch):
    t1, t2, y, cat_w = zip(*batch)

    enc = tokenizer(
        list(t1),
        list(t2),
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    return (
        enc,
        torch.tensor(y, dtype=torch.float32),
        torch.tensor(cat_w, dtype=torch.float32),
    )

def confidence_weight(y):
    if not USE_CONFIDENCE_WEIGHT:
        return torch.ones_like(y)

    confidence = torch.abs(2.0 * y - 1.0)
    return CONFIDENCE_FLOOR + (1.0 - CONFIDENCE_FLOOR) * confidence.pow(CONFIDENCE_POWER)

def weighted_soft_bce(logits, y, cat_w):
    per_example = F.binary_cross_entropy_with_logits(
        logits,
        y,
        reduction="none",
    )

    w = cat_w
    if USE_CONFIDENCE_WEIGHT:
        w = w * confidence_weight(y)

    # Normalize inside batch so LR scale stays stable.
    return (per_example * w).sum() / w.sum().clamp_min(1e-6)

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

## 7. Метрика и стабильный fast validation

В старом CE fast validation был обычным random sample. Для macro-метрики лучше, чтобы каждая категория была гарантированно представлена.

Берём до `FAST_VAL_PER_CATEGORY` примеров **на категорию**.

In [8]:
def macro_pr_auc(pairs_df, preds):
    z = pairs_df[["category", "target"]].reset_index(drop=True).copy()
    z["pred"] = np.asarray(preds)

    rows = []
    for cat, g in z.groupby("category", observed=True):
        y = g["target"].to_numpy()
        p = g["pred"].to_numpy()

        # AP is meaningful when validation category has positives.
        if y.sum() == 0:
            ap = np.nan
        else:
            ap = average_precision_score(y, p)

        rows.append({
            "category": cat,
            "pairs": len(g),
            "positive_rate": float(y.mean()),
            "PR_AUC": ap,
        })

    table = pd.DataFrame(rows).sort_values("PR_AUC").reset_index(drop=True)
    macro = float(table["PR_AUC"].dropna().mean())

    return macro, table

def make_balanced_fast_val(df, per_category=FAST_VAL_PER_CATEGORY, seed=0):
    parts = []
    for _, g in df.groupby("category", observed=True):
        n = min(per_category, len(g))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=True)

ml_val_fast = make_balanced_fast_val(ml_val)
print("fast val:", len(ml_val_fast))
display(
    ml_val_fast.groupby("category", observed=True)
    .size()
    .rename("pairs")
    .reset_index()
)

fast val: 56275


,category,pairs
0,Автотовары,3000
1,Аптека,3000
2,Бытовая техника,3000
3,Бытовая химия,2266
4,Галантерея и аксессуары,3000
5,Детские товары,3000
6,Дом и сад,3000
7,Канцелярские товары,3000
8,Красота и гигиена,3000
9,Мебель,3000


## 8. Модель

`multilingual-e5-small` используется здесь как backbone для **pair sequence classification**, а не как обычный bi-encoder.

Для classification head задаём больший learning rate, чем для backbone.

In [9]:
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
        attn_implementation="sdpa",
    )
except Exception as e:
    print("SDPA init fallback:", repr(e))
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1,
    )

model = model.to(device)

if USE_GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()

n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M")

# Differential LR: random classification head learns faster than pretrained backbone.
head_names = ("classifier", "score", "classification_head")

backbone_params = []
head_params = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue

    if any(h in name.lower() for h in head_names):
        head_params.append(p)
    else:
        backbone_params.append(p)

print("backbone tensors:", len(backbone_params))
print("head tensors:", len(head_params))
assert len(head_params) > 0, "Не удалось найти classification head."

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


parameters: 117.7M
backbone tensors: 199
head tensors: 2


In [10]:
@torch.no_grad()
def predict(model, pairs_df, batch_size=PRED_BATCH):
    model.eval()

    dl = DataLoader(
        PairDataset(pairs_df, training=False),
        batch_size=batch_size,
        collate_fn=collate,
        num_workers=0,
        shuffle=False,
        pin_memory=True,
    )

    out = []

    for enc, _, _ in dl:
        enc = {
            k: v.to(device, non_blocking=True)
            for k, v in enc.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(**enc).logits.squeeze(-1)

        out.append(torch.sigmoid(logits.float()).cpu().numpy())

    return np.concatenate(out)

## 9. Training stage-A

Особенности:
- 1 epoch на всех LLM-парах;
- gradient accumulation;
- soft labels;
- category-balanced loss;
- confidence weighting;
- random swap;
- gradient clipping;
- лучший checkpoint выбирается по **balanced LLM group holdout macro PR-AUC**.

Мы не запускаем manual stage-B.

In [11]:
train_ds = PairDataset(ml_train, training=True)

train_dl = DataLoader(
    train_ds,
    batch_size=MICRO_BATCH,
    collate_fn=collate,
    num_workers=0,
    shuffle=True,
    drop_last=True,
    pin_memory=True,
)

micro_steps_per_epoch = len(train_dl)
opt_steps_per_epoch = math.ceil(micro_steps_per_epoch / GRAD_ACCUM)
total_opt_steps = opt_steps_per_epoch * EPOCHS
warmup_steps = max(100, int(total_opt_steps * WARMUP_RATIO))

optimizer = torch.optim.AdamW(
    [
        {
            "params": backbone_params,
            "lr": BACKBONE_LR,
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": head_params,
            "lr": HEAD_LR,
            "weight_decay": WEIGHT_DECAY,
        },
    ]
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
)

scaler = torch.amp.GradScaler("cuda")

print("micro steps / epoch:", micro_steps_per_epoch)
print("optimizer steps / epoch:", opt_steps_per_epoch)
print("total optimizer steps:", total_opt_steps)
print("warmup steps:", warmup_steps)

micro steps / epoch: 342199
optimizer steps / epoch: 42775
total optimizer steps: 42775
warmup steps: 1283


In [12]:
def save_checkpoint(path, model, metric, opt_step, extra=None):
    payload = {
        "model": model.state_dict(),
        "metric": float(metric),
        "opt_step": int(opt_step),
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
    }
    if extra:
        payload.update(extra)
    torch.save(payload, path)

def load_checkpoint_model(path, model):
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    print(
        f"Loaded checkpoint: step={ckpt.get('opt_step')} "
        f"metric={ckpt.get('metric')}"
    )
    return ckpt

best_metric = -1.0
best_step = -1
global_opt_step = 0

t0 = time.time()
loss_sum = 0.0
loss_count = 0

optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    print(f"\n===== EPOCH {epoch + 1}/{EPOCHS} =====", flush=True)

    for micro_step, (enc, y, cat_w) in enumerate(train_dl, start=1):
        model.train()

        enc = {
            k: v.to(device, non_blocking=True)
            for k, v in enc.items()
        }
        y = y.to(device, non_blocking=True)
        cat_w = cat_w.to(device, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        ):
            logits = model(**enc).logits.squeeze(-1)
            loss = weighted_soft_bce(logits, y, cat_w)
            loss_for_backward = loss / GRAD_ACCUM

        scaler.scale(loss_for_backward).backward()

        loss_sum += float(loss.item())
        loss_count += 1

        do_step = (
            micro_step % GRAD_ACCUM == 0
            or micro_step == len(train_dl)
        )

        if not do_step:
            continue

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        global_opt_step += 1

        if global_opt_step % 1000 == 0:
            elapsed = time.time() - t0
            seen_pairs = micro_step * MICRO_BATCH
            pairs_per_sec = seen_pairs / max(elapsed, 1e-6)
            eta_h = (
                (total_opt_steps - global_opt_step)
                * (elapsed / global_opt_step)
                / 3600
            )

            print(
                f"step {global_opt_step:6d}/{total_opt_steps} "
                f"loss={loss_sum/max(loss_count,1):.5f} "
                f"{pairs_per_sec:.0f} pair/s "
                f"eta={eta_h:.2f}h",
                flush=True,
            )

            loss_sum = 0.0
            loss_count = 0

        should_eval = (
            global_opt_step % EVAL_EVERY_OPT_STEPS == 0
            or global_opt_step == total_opt_steps
        )

        if should_eval:
            val_pred = predict(model, ml_val_fast)
            val_macro, val_table = macro_pr_auc(ml_val_fast, val_pred)

            print(
                f"\n[VAL] step={global_opt_step} "
                f"LLM balanced-fast Macro PR-AUC={val_macro:.6f}",
                flush=True,
            )
            display(val_table)

            if val_macro > best_metric:
                best_metric = val_macro
                best_step = global_opt_step

                save_checkpoint(
                    BEST_CKPT,
                    model,
                    metric=best_metric,
                    opt_step=best_step,
                    extra={
                        "category_weight": category_weight,
                    },
                )

                print(
                    f"[BEST] saved {BEST_CKPT} "
                    f"metric={best_metric:.6f}",
                    flush=True,
                )

            gc.collect()
            torch.cuda.empty_cache()

print(
    f"\nTraining done in {(time.time()-t0)/3600:.2f}h. "
    f"Best fast macro={best_metric:.6f} at step={best_step}"
)


===== EPOCH 1/1 =====


step   1000/42775 loss=0.50399 213 pair/s eta=13.94h


step   2000/42775 loss=0.40001 213 pair/s eta=13.61h


step   3000/42775 loss=0.38219 213 pair/s eta=13.27h


step   4000/42775 loss=0.37349 213 pair/s eta=12.94h


...

[BEST] saved /kaggle/working/e5_macro_v2_best.pt metric=0.786379


step  31000/42775 loss=0.32648 211 pair/s eta=3.97h


step  32000/42775 loss=0.32672 211 pair/s eta=3.63h


step  33000/42775 loss=0.32695 211 pair/s eta=3.29h


step  34000/42775 loss=0.32648 211 pair/s eta=2.96h


## 10. Reload best checkpoint и считаем полные holdouts

Это важно: финальная модель — **не обязательно последний optimizer step**.

LLM holdout является основным. Manual holdout выводится только для диагностики.

In [ ]:
assert os.path.exists(BEST_CKPT), "Best checkpoint не найден."

best_info = load_checkpoint_model(BEST_CKPT, model)
model = model.to(device)

# Full LLM confident group holdout
t_eval = time.time()
llm_pred = predict(model, ml_val)
llm_macro, llm_table = macro_pr_auc(ml_val, llm_pred)

print("=" * 90)
print(f"FULL LLM GROUP HOLDOUT MACRO PR-AUC: {llm_macro:.6f}")
print(f"eval time: {(time.time()-t_eval)/60:.1f} min")
print("=" * 90)
display(llm_table)

# Manual diagnostic only
t_eval = time.time()
manual_pred = predict(model, m_val)
manual_macro, manual_table = macro_pr_auc(m_val, manual_pred)

print("=" * 90)
print(f"MANUAL GROUP HOLDOUT MACRO PR-AUC (diagnostic only): {manual_macro:.6f}")
print(f"eval time: {(time.time()-t_eval)/60:.1f} min")
print("=" * 90)
display(manual_table)

## 11. Сравнение с командным ориентиром

По `FOR_MISHA.md`:
- текущий лучший LB ≈ 0.450;
- `rubert-tiny2 stage-A` — текущий reference;
- ручной holdout нельзя использовать как proxy leaderboard;
- слабые категории: fashion.

В этой ячейке просто печатаем ключевые категории и ориентировочный signal.  
Формулу `LB ≈ 0.6 × llm-holdout` рассматриваем **только как грубую командную эвристику**, а не как гарантию.

In [ ]:
weak_candidates = {
    "Обувь",
    "Одежда",
    "Галантерея",
    "Ювелирные украшения",
    "Ювелирка",
}

print("LLM full macro:", round(llm_macro, 6))
print("rough team heuristic LB ~", round(0.6 * llm_macro, 4))
print("\nWeak/fashion categories:")
display(
    llm_table[
        llm_table["category"].isin(weak_candidates)
    ].sort_values("PR_AUC")
)

print("\nAll categories:")
display(llm_table)

## 12. Опциональный fashion continuation

**По умолчанию выключено.**

Идея: если E5 уже хорошо поднял общий LLM holdout, можно сделать короткий low-LR continuation только на слабых fashion LLM-парах и принять его **только если полный macro holdout реально улучшился**.

Это не manual fine-tune. Используются те же LLM labels и тот же test-like distribution.

Если времени сегодня мало — пропусти этот блок и используй best stage-A.

In [ ]:
RUN_FASHION_CONTINUATION = False

FASHION_LR = 8e-6
FASHION_MAX_OPT_STEPS = 4000
FASHION_EVAL_EVERY = 1000

if RUN_FASHION_CONTINUATION:
    fashion_mask = np.fromiter(
        (item_cat[i] in FASHION_CATEGORIES for i in ml_train.id1.values),
        dtype=bool,
        count=len(ml_train),
    )

    fashion_train = ml_train[fashion_mask].copy()
    print("fashion train pairs:", len(fashion_train))

    # Start from the best global checkpoint.
    load_checkpoint_model(BEST_CKPT, model)
    model = model.to(device)

    fashion_ds = PairDataset(fashion_train, training=True)
    fashion_dl = DataLoader(
        fashion_ds,
        batch_size=MICRO_BATCH,
        collate_fn=collate,
        num_workers=0,
        shuffle=True,
        drop_last=True,
        pin_memory=True,
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=FASHION_LR,
        weight_decay=WEIGHT_DECAY,
    )
    scaler_f = torch.amp.GradScaler("cuda")

    best_fashion_macro = llm_macro
    best_fashion_path = "/kaggle/working/e5_macro_v2_fashion_best.pt"
    opt.zero_grad(set_to_none=True)

    fashion_opt_step = 0

    for micro_step, (enc, y, cat_w) in enumerate(fashion_dl, start=1):
        model.train()

        enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
        y = y.to(device, non_blocking=True)
        cat_w = cat_w.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**enc).logits.squeeze(-1)
            loss = weighted_soft_bce(logits, y, cat_w) / GRAD_ACCUM

        scaler_f.scale(loss).backward()

        do_step = micro_step % GRAD_ACCUM == 0
        if not do_step:
            continue

        scaler_f.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler_f.step(opt)
        scaler_f.update()
        opt.zero_grad(set_to_none=True)

        fashion_opt_step += 1

        if (
            fashion_opt_step % FASHION_EVAL_EVERY == 0
            or fashion_opt_step >= FASHION_MAX_OPT_STEPS
        ):
            pred_f = predict(model, ml_val_fast)
            macro_f, table_f = macro_pr_auc(ml_val_fast, pred_f)

            print(
                f"[fashion] step={fashion_opt_step} "
                f"fast macro={macro_f:.6f}"
            )
            display(table_f)

            if macro_f > best_fashion_macro:
                best_fashion_macro = macro_f
                save_checkpoint(
                    best_fashion_path,
                    model,
                    metric=macro_f,
                    opt_step=fashion_opt_step,
                )
                print("saved fashion best:", best_fashion_path)

        if fashion_opt_step >= FASHION_MAX_OPT_STEPS:
            break

    # Full validation before accepting continuation.
    if os.path.exists(best_fashion_path):
        load_checkpoint_model(best_fashion_path, model)
        pred_f_full = predict(model, ml_val)
        macro_f_full, table_f_full = macro_pr_auc(ml_val, pred_f_full)

        print("base full macro:", llm_macro)
        print("fashion full macro:", macro_f_full)
        display(table_f_full)

        if macro_f_full > llm_macro:
            print("ACCEPT fashion continuation")
            shutil.copy2(best_fashion_path, BEST_CKPT)
            llm_macro = macro_f_full
            llm_table = table_f_full
        else:
            print("REJECT fashion continuation; restoring base best")
            load_checkpoint_model(BEST_CKPT, model)

    del fashion_train
    gc.collect()
    torch.cuda.empty_cache()
else:
    print("Fashion continuation skipped.")

## 13. Сохраняем финальную модель

На выходе получаем:
- `/kaggle/working/e5_macro_v2_final/`
- `/kaggle/working/e5_macro_v2_final.zip`
- `metrics.json`

Именно эту папку/архив затем можно передать человеку, который собирает inference-container.

In [ ]:
# Ensure model corresponds to accepted BEST_CKPT.
load_checkpoint_model(BEST_CKPT, model)
model = model.to("cpu")

os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

metrics = {
    "model": MODEL_NAME,
    "max_len": MAX_LEN,
    "max_attr_chars": MAX_ATTR_CHARS,
    "best_fast_llm_macro": float(best_info.get("metric", best_metric)),
    "best_step": int(best_info.get("opt_step", best_step)),
    "llm_group_holdout_macro": float(llm_macro),
    "manual_group_holdout_macro": float(manual_macro),
    "category_balance": USE_CATEGORY_BALANCE,
    "confidence_weight": USE_CONFIDENCE_WEIGHT,
    "confidence_floor": CONFIDENCE_FLOOR,
    "fashion_boost": FASHION_BOOST,
    "random_swap_prob": RANDOM_SWAP_PROB,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "effective_batch": EFFECTIVE_BATCH,
    "llm_by_category": {
        str(r["category"]): float(r["PR_AUC"])
        for _, r in llm_table.iterrows()
        if pd.notna(r["PR_AUC"])
    },
    "manual_by_category": {
        str(r["category"]): float(r["PR_AUC"])
        for _, r in manual_table.iterrows()
        if pd.notna(r["PR_AUC"])
    },
}

with open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

archive_base = "/kaggle/working/e5_macro_v2_final"
archive_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir="/kaggle/working",
    base_dir="e5_macro_v2_final",
)

print("saved model dir:", OUT_DIR)
print("saved archive:", archive_path)
print("archive size GB:", os.path.getsize(archive_path) / 1024**3)
print(json.dumps(metrics, ensure_ascii=False, indent=2)[:5000])

## 14. Что сообщить команде после выполнения

Скопируй в командный чат:

- backbone: `intfloat/multilingual-e5-small`
- stage: LLM only, 1 epoch
- max_len: 192
- objective: category-balanced + confidence-weighted soft BCE
- random swap: 0.5
- best optimizer step
- **full LLM group holdout Macro PR-AUC**
- 4 слабые fashion-категории по отдельности
- manual holdout только как диагностику
- путь к `e5_macro_v2_final.zip`

### Как принимать решение о сабмите

Приоритет:
1. сравнить **full LLM group holdout macro** с `rubert-tiny2 stage-A`;
2. посмотреть, выросли ли Обувь / Одежда / Галантерея / Ювелирка;
3. только после этого тратить один из командных сабмитов.

Если E5 даёт заметный прирост на LLM holdout, следующий логичный ход — не новый train с нуля, а inference benchmark на H100 / контейнер и затем rank-average с независимым GBM/CE решением.